In [7]:
!pip install -q -U datasets pandas scikit-learn trl peft bitsandbytes accelerate transformers librosa

import pandas as pd
import numpy as np
import librosa
import json
from huggingface_hub import hf_hub_download
from datasets import load_dataset, Dataset, Audio, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 110.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

ERROR: Operation cancelled by user

In [ ]:
# ==========================================
# CENTRALIZED DATA PIPELINE CONFIGURATION
# ==========================================
class DataConfig:
    # --- API & Authentication ---
    HF_TOKEN = "put_token_here"

    # --- Module A: Text Pipeline (Hindi-to-Haryanvi) ---
    HF_REPO_ID = "Satyam-Srivastava/RDS"
    HF_REPO_TYPE = "dataset"
    TXT_HINDI_FILE = "text_data/hindi1.txt"
    TXT_HARYANVI_FILE = "text_data/haryanvi1.txt"
    JSON_FILE = "text_data/hindi_haryanvi_dataset_5000.json"

    TEXT_DATASET_PATH = "synthetic_haryanvi_corpus.json"
    TEXT_MIN_WORDS = 3
    TEXT_MAX_WORDS = 25
    LLM_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    LLM_MAX_LENGTH = 128

    # --- Module B: Audio Pipeline (TTS) ---
    TTS_DATASET_ID = "ankitdhiman/haryanvi-tts"
    TARGET_SR = 22050     # Standard for VITS
    AUDIO_MIN_DURATION = 1.0  # seconds
    AUDIO_MAX_DURATION = 15.0 # seconds

    # --- Splitting & Reproducibility ---
    RANDOM_SEED = 42
    TEXT_TEST_SPLIT = 0.20  # Leaves 80% for train, 20% for val/test
    TEXT_VAL_SPLIT = 0.50   # Splits the 20% equally into 10% val, 10% test
    AUDIO_EVAL_SPLIT = 0.10 # 90/10 split for VITS training

config = DataConfig()

In [3]:
# ==========================================
# MODULE A: TEXT DATA INGESTION & QUALITY ASSESSMENT
# ==========================================
print("--- Downloading and Loading Text Datasets ---")

# 1. Download and parse the parallel text files
print("Fetching hindi1.txt and haryanvi1.txt...")
hindi_txt_path = hf_hub_download(repo_id=config.HF_REPO_ID, filename=config.TXT_HINDI_FILE, repo_type=config.HF_REPO_TYPE, token=config.HF_TOKEN)
haryanvi_txt_path = hf_hub_download(repo_id=config.HF_REPO_ID, filename=config.TXT_HARYANVI_FILE, repo_type=config.HF_REPO_TYPE, token=config.HF_TOKEN)

with open(hindi_txt_path, 'r', encoding='utf-8') as f:
    hindi_lines = [line.strip() for line in f if line.strip()]

with open(haryanvi_txt_path, 'r', encoding='utf-8') as f:
    haryanvi_lines = [line.strip() for line in f if line.strip()]

# Zip them into a DataFrame (using the minimum length in case of a missing line mismatch)
min_len = min(len(hindi_lines), len(haryanvi_lines))
df_txt = pd.DataFrame({
    "hindi": hindi_lines[:min_len],
    "haryanvi": haryanvi_lines[:min_len]
})
print(f"Loaded {len(df_txt)} pairs from text files.")

# 2. Download and parse the JSON file
print("Fetching hindi_haryanvi_dataset_5000.json...")
json_path = hf_hub_download(repo_id=config.HF_REPO_ID, filename=config.JSON_FILE, repo_type=config.HF_REPO_TYPE, token=config.HF_TOKEN)

with open(json_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# Convert JSON to DataFrame and normalize the capitalized column names
df_json = pd.DataFrame(json_data)
df_json.rename(columns={"Hindi": "hindi", "Haryanvi": "haryanvi"}, inplace=True)
print(f"Loaded {len(df_json)} pairs from JSON file.")

# 3. Combine both dataframes
df_text = pd.concat([df_txt, df_json], ignore_index=True)
print(f"\nInitial combined dataset size: {len(df_text)}")

print("\n--- Quality Assessment ---")
missing_values = df_text.isnull().sum()
print(f"Missing values:\n{missing_values}")

initial_len = len(df_text)
df_text = df_text.drop_duplicates()
print(f"Removed {initial_len - len(df_text)} duplicate rows to prevent leakage.")

# Filter out extreme outliers to maintain quality
df_text['hindi_len'] = df_text['hindi'].apply(lambda x: len(x.split()))
df_text_clean = df_text[
    (df_text['hindi_len'] >= config.TEXT_MIN_WORDS) &
    (df_text['hindi_len'] <= config.TEXT_MAX_WORDS)
].copy()

# ==========================================
# TRAIN / VAL / TEST SPLIT
# ==========================================
train_df, temp_df = train_test_split(
    df_text_clean,
    test_size=config.TEXT_TEST_SPLIT,
    random_state=config.RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=config.TEXT_VAL_SPLIT,
    random_state=config.RANDOM_SEED
)

print(f"\n--- Split Sizes ---")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

text_dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False)
})

--- Downloading and Loading Text Datasets ---
Fetching hindi1.txt and haryanvi1.txt...


hindi1.txt: 0.00B [00:00, ?B/s]

haryanvi1.txt: 0.00B [00:00, ?B/s]

Loaded 25 pairs from text files.
Fetching hindi_haryanvi_dataset_5000.json...


hindi_haryanvi_dataset_5000.json: 0.00B [00:00, ?B/s]

Loaded 5000 pairs from JSON file.

Initial combined dataset size: 5025

--- Quality Assessment ---
Missing values:
hindi       0
haryanvi    0
dtype: int64
Removed 8 duplicate rows to prevent leakage.

--- Split Sizes ---
Train: 4011 | Val: 501 | Test: 502


In [4]:
# ==========================================
# LLM FORMATTING & TOKENIZATION PIPELINE
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(config.LLM_MODEL_ID, token=config.HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

def format_for_llama(examples):
    formatted_texts = []
    for hindi, haryanvi in zip(examples['hindi'], examples['haryanvi']):
        dialog = [
            {"role": "system", "content": "You are a precise Hindi to Haryanvi (Bangru) translator. Convert the standard Hindi input into grammatically correct Haryanvi. Follow systematic lexical substitutions and morpho-syntactic rules."},
            {"role": "user", "content": hindi},
            {"role": "assistant", "content": haryanvi}
        ]
        formatted_texts.append(tokenizer.apply_chat_template(dialog, tokenize=False))

    tokenized = tokenizer(
        formatted_texts,
        max_length=config.LLM_MAX_LENGTH,
        padding="max_length",
        truncation=True
    )
    return tokenized

llm_ready_dataset = text_dataset_dict.map(
    format_for_llama,
    batched=True,
    remove_columns=['hindi', 'haryanvi', 'hindi_len']
)
print("\nLLM Dataset properly formatted with Chat Templates.")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/4011 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

Map:   0%|          | 0/502 [00:00<?, ? examples/s]


LLM Dataset properly formatted with Chat Templates.


In [ ]:
# ==========================================
# MODULE B: AUDIO DATASET (TTS) PREPARATION
# ==========================================
print(f"Loading {config.TTS_DATASET_ID}...")
audio_dataset = load_dataset(config.TTS_DATASET_ID, split="train", token=config.HF_TOKEN)

# Force resampling to maintain consistency
audio_dataset = audio_dataset.cast_column("audio", Audio(sampling_rate=config.TARGET_SR))

def calculate_audio_duration(example):
    audio_array = example['audio']['array']
    sr = example['audio']['sampling_rate']
    example['duration'] = len(audio_array) / sr
    return example

audio_dataset = audio_dataset.map(calculate_audio_duration)

# Filter out extreme outliers to prevent VRAM spikes and bad alignment
filtered_audio = audio_dataset.filter(
    lambda x: config.AUDIO_MIN_DURATION <= x['duration'] <= config.AUDIO_MAX_DURATION
)
print(f"Retained {len(filtered_audio)} audio samples after duration filtering.")

# Train / Val Split for Audio
audio_dataset_dict = filtered_audio.train_test_split(
    test_size=config.AUDIO_EVAL_SPLIT,
    seed=config.RANDOM_SEED
)

def prepare_vits_dataset(examples):
    return {
        "audio_array": [audio["array"] for audio in examples["audio"]],
        "normalized_text": [text.strip() for text in examples["transcript"]]
    }

final_tts_dataset = audio_dataset_dict.map(
    prepare_vits_dataset,
    batched=True,
    remove_columns=audio_dataset_dict['train'].column_names
)

print("\n--- Audio Pipeline Ready ---")
print(f"Audio Train Split: {len(final_tts_dataset['train'])} samples")
print(f"Audio Eval Split: {len(final_tts_dataset['test'])} samples")

In [5]:
import torch
import os
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from trl import SFTTrainer

In [9]:
# ==========================================
# 1. A100-OPTIMIZED MODEL LOADING
# ==========================================
print("\n--- Initializing A100 Optimized QLoRA ---")

# A100 heavily benefits from bfloat16 and nested 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading Base Model: {config.LLM_MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    config.LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=config.HF_TOKEN,
    attn_implementation="sdpa"
)

# Enable gradient checkpointing to save VRAM during backpropagation
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


--- Initializing A100 Optimized QLoRA ---
Loading Base Model: meta-llama/Meta-Llama-3.1-8B-Instruct...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [10]:
# ==========================================
# 2. LORA ADAPTER CONFIGURATION
# ==========================================
# Targeting all linear layers ensures the model can learn complex
# morpho-syntactic dialect shifts, not just basic vocabulary routing.
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [16]:
# Make sure to import SFTConfig from trl
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling

# ==========================================
# 3. TRAINING LOOP (SFTTrainer)
# ==========================================
OUTPUT_DIR = "./llama-3.1-haryanvi-adapter"

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False # False because this is causal LM, not masked LM
)

# Replace TrainingArguments with SFTConfig
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,        # Effective batch size = 16
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    max_grad_norm=0.3,
    warmup_steps=50,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=3,
    optim="paged_adamw_32bit",
    fp16=False,
    bf16=True,
    report_to="none",
    max_length=config.LLM_MAX_LENGTH,
    dataset_text_field=None               # Ignored since data is pre-tokenized
)

print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=llm_ready_dataset["train"],
    eval_dataset=llm_ready_dataset["validation"],
    args=training_args,
    data_collator=data_collator
)

Initializing SFTTrainer...


Truncating train dataset:   0%|          | 0/4011 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/501 [00:00<?, ? examples/s]

In [17]:
print("Starting Training Loop...")
trainer.train()

print(f"Saving Final Adapter Weights to {OUTPUT_DIR}-final...")
trainer.model.save_pretrained(f"{OUTPUT_DIR}-final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}-final")

print("Pipeline Complete. Text translation module is ready for inference.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Starting Training Loop...


Epoch,Training Loss,Validation Loss
1,0.094099,0.095431
2,0.094049,0.093108
3,0.090524,0.092308


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error (Request ID: Root=1-69bbfddd-15bd353363a4905d476b5111;0cc723b3-e003-4801-b5d0-b126da09744d)

403 Forbidden: Please enable access to public gated repositories in your fine-grained token settings to view this repository..
Cannot access content at: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Make sure your token has the correct permissions. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3.1-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error (Request 

Saving Final Adapter Weights to ./llama-3.1-haryanvi-adapter-final...


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error (Request ID: Root=1-69bc039d-7d7cd7d77401533c53f45f7f;cb4bc1cf-e072-48f9-8a6c-35d2ba84cf69)

403 Forbidden: Please enable access to public gated repositories in your fine-grained token settings to view this repository..
Cannot access content at: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Make sure your token has the correct permissions. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B-Instruct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3.1-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


Pipeline Complete. Text translation module is ready for inference.


In [22]:
def translate_to_haryanvi(hindi_text):
    messages = [
        {"role": "system", "content": "You are a precise Hindi to Haryanvi (Bangru) translator. Convert the standard Hindi input into grammatically correct Haryanvi. Follow systematic lexical substitutions and morpho-syntactic rules."},
        {"role": "user", "content": hindi_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.15
        )

    # 1. Decode EVERYTHING, including special tokens
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # 2. Split exactly where the model's answer begins
    assistant_tag = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    if assistant_tag in full_text:
        translation = full_text.split(assistant_tag)[-1]
    else:
        translation = full_text

    # 3. Clean up the end-of-turn tokens
    translation = translation.replace("<|eot_id|>", "").replace("<|eos_id|>", "").strip()

    # --- AGGRESSIVE POST-PROCESSING CUTOFF ---
    # Force the string to end at the very first sentence terminator or hallucination
    translation = translation.split("\n")[0]       # Cut off at the first new line
    translation = translation.split("|")[0]        # Cut off at the first pipe
    translation = translation.split("स्कीम")[0]    # Cut off if it starts explaining the "schema"

    # Ensure it ends with a clean Hindi Danda
    translation = translation.split("।")[0].strip() + "।"

    return translation

# ==========================================
# TEST YOUR TRANSLATOR
# ==========================================
print("\n--- Testing Haryanvi Translation ---")

test_sentences = [
    "मैं आज बाजार जा रहा हूँ।",
    "क्या तुम कल स्कूल जाओगे?",
    "यह किताब मेरी है, इसे मत छूना।"
]

for sentence in test_sentences:
    print(f"Standard Hindi: {sentence}")
    print(f"Bangru Output:  {translate_to_haryanvi(sentence)}\n")


--- Testing Haryanvi Translation ---
Standard Hindi: मैं आज बाजार जा रहा हूँ।
Bangru Output:  मैं आज बाजार जा रया सै।

Standard Hindi: क्या तुम कल स्कूल जाओगे?
Bangru Output:  कया तुम कल स्कूल जावोगे।

Standard Hindi: यह किताब मेरी है, इसे मत छूना।
Bangru Output:  या किताब मेरी सै, इसे न छूणा।



In [24]:
import os
import shutil

# ==========================================
# 1. SAVE FINAL WEIGHTS
# ==========================================
FINAL_SAVE_DIR = "./llama-3.1-haryanvi-adapter-final"

print(f"--- Saving Final Adapter to {FINAL_SAVE_DIR} ---")
# This saves the final LoRA weights, not the massive base model
trainer.save_model(FINAL_SAVE_DIR)
# Save the tokenizer so you have the exact chat_template formatting
tokenizer.save_pretrained(FINAL_SAVE_DIR)
print("✅ Weights and tokenizer saved successfully.")

# ==========================================
# 2. ZIP AND DOWNLOAD
# ==========================================
# Naming the zip file for your Gyan Align project
zip_filename = "rds_lora_adapter"

print(f"\n--- Zipping files into {zip_filename}.zip ---")
shutil.make_archive(zip_filename, 'zip', FINAL_SAVE_DIR)
print("✅ Zip archive created.")

print("\n--- Triggering Download ---")
# If using Google Colab, this automatically prompts the browser download
try:
    from google.colab import files
    files.download(f"{zip_filename}.zip")
    print("Download started! Check your browser's download manager.")
except ImportError:
    # Fallback for local Jupyter or Kaggle environments
    from IPython.display import FileLink
    display(FileLink(f"{zip_filename}.zip"))
    print("Click the link above to download your adapter bundle.")

--- Saving Final Adapter to ./llama-3.1-haryanvi-adapter-final ---


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error (Request ID: Root=1-69bc0659-76cb53ac7e8637ce33a03c70;34f69284-85a8-4a74-985d-f4beadbaee5a)

403 Forbidden: Please enable access to public gated repositories in your fine-grained token settings to view this repository..
Cannot access content at: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Make sure your token has the correct permissions. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B-Instruct.
  warnings.warn(


✅ Weights and tokenizer saved successfully.

--- Zipping files into rds_lora_adapter.zip ---
✅ Zip archive created.

--- Triggering Download ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started! Check your browser's download manager.
